# Laboratorio 1 · ¿Qué sabe realmente esta base sobre nosotros?

**Explorar → Analizar → Experimentar → Decidir → Justificar → Reflexionar**

Leyes, Ética y Protección de Datos · Especialización en Análisis Estadístico para Ciencia de Datos · Docente: Wilson Sandoval Rodríguez

---

> **Versión con soluciones.** Material de apoyo docente. Incluye respuestas orientativas y las celdas ya ejecutadas.

> Todos los datos son **sintéticos**. Ninguna persona real está representada.


## Qué va a hacer aquí

| | |
|:--|:--|
| **Aprenderá a** | Clasificar variables por el riesgo que introducen y medir el riesgo de reidentificación de una base |
| **Duración** | 35 minutos |
| **Evidencia** | Sus respuestas a las 9 preguntas y la base minimizada que construya |
| **Unidad** | 1 · Fundamentos legales |

Este cuaderno **no es una clase de programación**. El código es corto a
propósito. Lo que se evalúa es la decisión que usted toma después de leer la
salida.

---

# 1 · EXPLORAR

*¿Qué hay en esta base?*

In [1]:
from pathlib import Path
import pandas as pd

# Funciona en tres sitios sin cambiar nada:
#   - dentro del repositorio (labs/ o raíz)
#   - en Google Colab
#   - en cualquier equipo con internet
RUTA = Path("../data/clientes_sinteticos.csv")
if not RUTA.exists():
    RUTA = Path("data/clientes_sinteticos.csv")
if not RUTA.exists():
    RUTA = "https://raw.githubusercontent.com/wilsonsr/leyes-etica-proteccion-datos/main/data/clientes_sinteticos.csv"

print("Origen de los datos:", RUTA)

Origen de los datos: ../data/clientes_sinteticos.csv


In [2]:
df = pd.read_csv(RUTA)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 130)

print(f"Registros: {len(df)}   Variables: {df.shape[1]}")
df.head(3)

Registros: 1512   Variables: 35


,cliente_id,nombre,email,telefono,documento,fecha_nacimiento,edad,sexo,ciudad,barrio,direccion,latitud,longitud,estrato,ocupacion,ingresos_mensuales,dispositivo,ip,pais_residencia,canal_adquisicion,origen_dato,fecha_autorizacion,consentimiento_marketing,visitas_web,productos_vistos,categoria_top,compras_6m,monto_compras,fecha_ultima_compra,compras_farmacia_6m,entrega_asistida,busquedas_maternidad,score_riesgo_interno,segmento,churn
0,DM-00865,Luis Castro,luis.castro806@mail.co,3361052437,SYN-92480821,1992-12-31,33,M,Barranquilla,Simon Bolivar,Carrera 7 # 58-18,10.97607,-74.80194,2.0,Pensionado/a,2460000.0,Android,135.169.205.166,Colombia,email_marketing,lista_comprada_tercero,NaN,0,12,36,moda,2,418000.0,2026-08-29,0,0,0,26,fiel,0
1,DM-00714,Paula Beltran,paula.beltran226@correo.com,3552777695,SYN-95884611,1974-01-01,52,F,Cartagena,Manga,Carrera 53 # 50-88,10.33459,-75.45909,1.0,Comerciante,1710000.0,iOS,192.175.234.41,Colombia,publicidad_pagada,registro_cuenta,2024-02-23,1,9,28,supermercado,4,778000.0,2026-05-11,1,0,0,48,estable,0
2,DM-00572,Alvaro Riveros,alvaro.riveros457@mail.co,3693537285,SYN-35266492,2000-12-31,25,F,Barranquilla,El Prado,Diagonal 42 # 18-58,10.95000,-74.77236,3.0,Comerciante,2950000.0,Android,57.79.150.169,Alemania,publicidad_pagada,programa_fidelizacion,2025-05-14,1,12,19,tecnologia,2,435000.0,2026-07-26,0,0,0,41,estable,0


::: {.rds-card .decide}
**Pregunta 1.** Mire las tres primeras filas. Sin contar todavía las columnas: ¿cuántas de las que alcanza a ver le permitirían, por sí solas, llamar por teléfono a esa persona?
:::

::: {.callout-note collapse="true"}
## Respuesta orientativa (versión docente)

Cinco: `nombre`, `email`, `telefono`, `documento` y `direccion`. La respuesta importa menos que la reacción: casi nadie espera encontrar cinco identificadores directos en una base «de comportamiento de compra».
:::

In [3]:
for i, col in enumerate(df.columns, start=1):
    print(f"{i:>2}. {col}")

 1. cliente_id
 2. nombre
 3. email
 4. telefono
 5. documento
 6. fecha_nacimiento
 7. edad
 8. sexo
 9. ciudad
10. barrio
11. direccion
12. latitud
13. longitud
14. estrato
15. ocupacion
16. ingresos_mensuales
17. dispositivo
18. ip
19. pais_residencia
20. canal_adquisicion
21. origen_dato
22. fecha_autorizacion
23. consentimiento_marketing
24. visitas_web
25. productos_vistos
26. categoria_top
27. compras_6m
28. monto_compras
29. fecha_ultima_compra
30. compras_farmacia_6m
31. entrega_asistida
32. busquedas_maternidad
33. score_riesgo_interno
34. segmento
35. churn


In [4]:
resumen = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "no_nulos": df.notna().sum(),
    "distintos": df.nunique(),
})
resumen["% distintos"] = (resumen["distintos"] / len(df) * 100).round(1)
resumen.sort_values("% distintos", ascending=False).head(12)

,tipo,no_nulos,distintos,% distintos
cliente_id,str,1512,1512,100.0
email,str,1512,1500,99.2
telefono,str,1512,1500,99.2
direccion,str,1512,1500,99.2
documento,str,1512,1500,99.2
ip,str,1512,1500,99.2
latitud,float64,1512,1488,98.4
longitud,float64,1512,1483,98.1
fecha_autorizacion,str,1276,867,57.3
nombre,str,1512,769,50.9


::: {.rds-card .decide}
**Pregunta 2.** Las variables con un porcentaje de valores distintos cercano al 100 % son casi siempre identificadores. ¿Cuáles aparecen arriba y cuál de ellas *no* esperaba encontrar ahí?
:::

::: {.callout-note collapse="true"}
## Respuesta orientativa (versión docente)

`cliente_id`, `documento`, `direccion`, `email`, `telefono` y también `ip`. La que sorprende es `ip`: es un identificador en línea, se trata como dato personal y casi nunca se documenta como tal.
:::

---

# 2 · ANALIZAR

*¿Qué significa lo que hay?*

### Los faltantes no siempre son un problema de imputación

La pregunta útil no es *cuántos* faltan, sino *por qué* faltan.

In [5]:
faltantes = (
    df.isna().sum().loc[lambda s: s > 0]
      .sort_values(ascending=False).to_frame("faltantes")
)
faltantes["%"] = (faltantes["faltantes"] / len(df) * 100).round(1)
faltantes

,faltantes,%
fecha_autorizacion,236,15.6
ingresos_mensuales,135,8.9
ocupacion,76,5.0
estrato,60,4.0


In [6]:
# ¿Los faltantes de fecha_autorizacion están repartidos al azar?
pd.crosstab(
    df["origen_dato"],
    df["fecha_autorizacion"].isna().map({True: "sin fecha", False: "con fecha"}),
)

fecha_autorizacion,con fecha,sin fecha
origen_dato,,
enriquecimiento_web,0,92
entrega_domicilio,348,0
facturacion,464,0
lista_comprada_tercero,0,144
programa_fidelizacion,175,0
registro_cuenta,289,0


::: {.rds-card .riesgo}
**Pregunta 3.** Los faltantes de `fecha_autorizacion` no están repartidos al azar: se concentran en dos orígenes. ¿Cuáles? ¿Y qué significa, en términos prácticos, no tener fecha de autorización para esos registros?

Esto no es un problema de imputación. Es un problema de **evidencia**.
:::

::: {.callout-note collapse="true"}
## Respuesta orientativa (versión docente)

`lista_comprada_tercero` (144 registros) y `enriquecimiento_web` (92). Suman 236, el 15,6 % de la base. Significa que no se puede acreditar que esas personas hayan autorizado nada. Imputar esa fecha sería fabricar evidencia. Lo correcto es aislarlos, no borrarlos: borrarlos destruiría la prueba de que existieron.
:::

### Clasificar por riesgo, no por tipo

El tipo que infiere `pandas` no dice nada sobre el riesgo. `object` puede ser un
nombre propio o una categoría inofensiva.

| Etiqueta | Significa |
|:--|:--|
| `directo` | Identifica a la persona por sí solo |
| `indirecto` | Identifica vía dispositivo, cuenta o ubicación precisa |
| `cuasi` | Solo no identifica; combinado con otros, sí |
| `comportamiento` | Lo que la persona hizo |
| `inferido` | Lo que la empresa dedujo |
| `proxy_sensible` | No es sensible, pero permite inferir uno |
| `control` | Metadato del tratamiento (origen, autorización) |
| `objetivo` | La variable que queremos predecir |

In [7]:
clasificacion = {
    # identificadores directos
    "nombre": "directo", "email": "directo", "telefono": "directo",
    "documento": "directo", "direccion": "directo",
    # identificadores indirectos
    "cliente_id": "indirecto", "ip": "indirecto", "dispositivo": "indirecto",
    "latitud": "indirecto", "longitud": "indirecto",
    # cuasi-identificadores
    "fecha_nacimiento": "cuasi", "edad": "cuasi", "sexo": "cuasi",
    "ciudad": "cuasi", "barrio": "cuasi", "ocupacion": "cuasi",
    "estrato": "cuasi", "ingresos_mensuales": "cuasi",
    "pais_residencia": "cuasi",
    # comportamiento
    "visitas_web": "comportamiento", "productos_vistos": "comportamiento",
    "categoria_top": "comportamiento", "compras_6m": "comportamiento",
    "monto_compras": "comportamiento", "fecha_ultima_compra": "comportamiento",
    "canal_adquisicion": "comportamiento",
    # proxies de datos sensibles
    "compras_farmacia_6m": "proxy_sensible",
    "entrega_asistida": "proxy_sensible",
    "busquedas_maternidad": "proxy_sensible",
    # datos inferidos por la empresa
    "score_riesgo_interno": "inferido", "segmento": "inferido",
    # metadatos de tratamiento
    "origen_dato": "control", "fecha_autorizacion": "control",
    "consentimiento_marketing": "control",
    # objetivo
    "churn": "objetivo",
}

faltan = set(df.columns) - set(clasificacion)
print("Variables sin clasificar:", sorted(faltan) or "ninguna")

Variables sin clasificar: ninguna


In [8]:
mapa = pd.Series(clasificacion, name="categoria").rename_axis("variable")
mapa.value_counts().to_frame("n_variables")

,n_variables
categoria,
cuasi,9
comportamiento,7
directo,5
indirecto,5
proxy_sensible,3
control,3
inferido,2
objetivo,1


::: {.rds-card .decide}
**Pregunta 4.** Esta clasificación es **discutible a propósito**. Elija dos variables que usted habría puesto en otra categoría y defienda el cambio.

Candidatas frecuentes: `latitud`/`longitud` (¿indirecto o cuasi?), `ingresos_mensuales` (¿cuasi o proxy sensible?), `categoria_top` (¿comportamiento o proxy sensible, cuando el valor es `salud_bienestar`?).
:::

::: {.callout-note collapse="true"}
## Respuesta orientativa (versión docente)

No hay una respuesta correcta; hay respuestas argumentadas. `latitud`/`longitud` con ruido de ±3 km funcionan más como cuasi-identificador que como identificador directo, pero combinadas con `barrio` se vuelven muy identificadoras. `ingresos_mensuales` es proxy socioeconómico y por tanto proxy de un atributo que puede discriminar. `categoria_top = salud_bienestar` es exactamente un proxy sensible. Lo que se evalúa es que el estudiante note que la categoría depende del **uso**, no solo del contenido.
:::

---

# 3 · EXPERIMENTAR

*¿Qué pasa si…?*

### El experimento central del laboratorio

Ninguna de estas columnas identifica sola. Veamos qué ocurre al combinarlas.

In [9]:
def riesgo_unicidad(datos, columnas):
    # Cuenta cuántos registros quedan SOLOS en su grupo (k = 1).
    grupos = datos.groupby(columnas, dropna=False, observed=True).size()
    unicos = int((grupos == 1).sum())
    return {
        "variables": " + ".join(columnas),
        "combinaciones": int(len(grupos)),
        "registros k=1": unicos,
        "% en riesgo": round(unicos / len(datos) * 100, 1),
        "k mínimo": int(grupos.min()),
    }


combinaciones = [
    ["ciudad"],
    ["ciudad", "sexo"],
    ["ciudad", "sexo", "edad"],
    ["ciudad", "sexo", "edad", "ocupacion"],
    ["ciudad", "barrio", "sexo", "edad"],
    ["ciudad", "barrio", "sexo", "edad", "ocupacion", "estrato"],
]

pd.DataFrame([riesgo_unicidad(df, c) for c in combinaciones])

,variables,combinaciones,registros k=1,% en riesgo,k mínimo
0,ciudad,12,0,0.0,42
1,ciudad + sexo,46,3,0.2,1
2,ciudad + sexo + edad,738,385,25.5,1
3,ciudad + sexo + edad + ocupacion,1385,1269,83.9,1
4,ciudad + barrio + sexo + edad,1194,945,62.5,1
5,ciudad + barrio + sexo + edad + ocupacion + es...,1496,1480,97.9,1


::: {.rds-card .riesgo}
**Pregunta 5.** Con una sola variable el riesgo es cero. ¿A partir de cuántas variables la mayoría de las personas de esta base queda sola en su grupo?

Anote el número. Es el argumento que va a necesitar la próxima vez que alguien diga «ya le quitamos los nombres».
:::

::: {.callout-note collapse="true"}
## Respuesta orientativa (versión docente)

Con cuatro variables (ciudad, sexo, edad, ocupación) el 83,9 % de los registros queda solo. Con seis, el 97,9 %. El salto grande ocurre al agregar `edad`, que por sí sola tiene 65 valores posibles.
:::

### Ahora al revés: reducir el riesgo y medirlo

La anonimización es un **resultado que se mide**, no una operación que se
aplica. Apliquemos tres generalizaciones y volvamos a medir.

In [10]:
caso = df[["edad", "sexo", "ciudad", "ocupacion"]].copy()
caso_v2 = caso.copy()

# (a) Edad en rangos de 5 años
caso_v2["edad"] = pd.cut(caso_v2["edad"], bins=range(15, 90, 5)).astype(str)

# (b) Ocupación en tres bloques amplios
grandes = {
    "Ingeniero/a": "técnico-profesional", "Tecnico/a": "técnico-profesional",
    "Disenador/a": "técnico-profesional", "Contador/a": "técnico-profesional",
    "Abogado/a": "técnico-profesional", "Administrador/a": "técnico-profesional",
    "Docente": "servicios", "Enfermero/a": "servicios",
    "Comerciante": "servicios", "Independiente": "servicios",
    "Estudiante": "sin actividad remunerada",
    "Pensionado/a": "sin actividad remunerada",
}
caso_v2["ocupacion"] = caso_v2["ocupacion"].map(grandes).fillna("otro")

# (c) Ciudad en regiones
regiones = {
    "Bogota": "Centro", "Ibague": "Centro", "Villavicencio": "Centro",
    "Medellin": "Antioquia-Eje", "Pereira": "Antioquia-Eje", "Manizales": "Antioquia-Eje",
    "Cali": "Pacífico", "Pasto": "Pacífico",
    "Barranquilla": "Caribe", "Cartagena": "Caribe", "Santa Marta": "Caribe",
    "Bucaramanga": "Nororiente",
}
caso_v2["ciudad"] = caso_v2["ciudad"].map(regiones).fillna("otro")

pd.DataFrame([
    {"versión": "original", **riesgo_unicidad(caso, ["edad", "sexo", "ciudad", "ocupacion"])},
    {"versión": "generalizada", **riesgo_unicidad(caso_v2, ["edad", "sexo", "ciudad", "ocupacion"])},
])

,versión,variables,combinaciones,registros k=1,% en riesgo,k mínimo
0,original,edad + sexo + ciudad + ocupacion,1385,1269,83.9,1
1,generalizada,edad + sexo + ciudad + ocupacion,406,160,10.6,1


::: {.rds-card .prueba}
**Pregunta 6.** El porcentaje en riesgo bajó. ¿Bajó lo suficiente?

No hay un umbral universal. Lo que sí hay es una obligación: **decir cuál es el umbral que se aceptó y por qué**. Escriba el suyo.

Y la contrapartida: ¿qué análisis dejó de ser posible con la base generalizada? ¿Vale la pena?
:::

::: {.callout-note collapse="true"}
## Respuesta orientativa (versión docente)

De 83,9 % a 10,6 %. Sigue habiendo 160 personas solas en su grupo, así que la base **no** es anónima: es menos riesgosa. Lo que se pierde: cualquier análisis por municipio, cualquier análisis de la relación edad–comportamiento con resolución fina y cualquier segmentación ocupacional específica. El criterio profesional es declarar el umbral (por ejemplo *k* ≥ 5 para el 95 % de los registros) y mostrar el número.
:::

---

# 4 · DECIDIR

*¿Qué hacemos?*

### Minimizar no es «quitar columnas»

Minimizar es responder, para cada columna, **por qué la necesito para esta
finalidad concreta**. La finalidad declarada aquí es: *predecir abandono para
una campaña de retención*.

In [11]:
variables_minimas = [
    "cliente_id",            # necesario para actuar sobre el cliente
    "visitas_web",
    "productos_vistos",
    "compras_6m",
    "monto_compras",
    "fecha_ultima_compra",
    "canal_adquisicion",
    "churn",
]

df_min = df[variables_minimas].copy()
print(f"Original: {df.shape[1]} variables  ->  Minimizada: {df_min.shape[1]}")
print(f"Reducción: {100 - df_min.shape[1] / df.shape[1] * 100:.0f} %")

Original: 35 variables  ->  Minimizada: 8
Reducción: 77 %


### Seudonimizar no es anonimizar

Reemplazamos el identificador por un hash. Es buena práctica **y no es
anonimización**: la celda siguiente muestra por qué.

In [12]:
import hashlib

SAL = "curso-lepd-2026"   # en producción: secreto, rotado y fuera del código


def seudonimizar(valor, sal=SAL, largo=12):
    return hashlib.sha256(f"{sal}{valor}".encode("utf-8")).hexdigest()[:largo].upper()


df_seudo = df_min.copy()
df_seudo["cliente_id"] = df_seudo["cliente_id"].map(seudonimizar)
df_seudo.head(4)

,cliente_id,visitas_web,productos_vistos,compras_6m,monto_compras,fecha_ultima_compra,canal_adquisicion,churn
0,D4C5AC809820,12,36,2,418000.0,2026-08-29,email_marketing,0
1,DBD3EE164CF6,9,28,4,778000.0,2026-05-11,publicidad_pagada,0
2,33E091D5A353,12,19,2,435000.0,2026-07-26,publicidad_pagada,0
3,ED774388A041,15,28,5,2586000.0,2026-03-30,marketplace,0


In [13]:
# La tabla de equivalencias: esto es lo que impide llamarlo anonimización.
tabla_equivalencias = pd.DataFrame({
    "cliente_id": df["cliente_id"],
    "seudonimo": df["cliente_id"].map(seudonimizar),
    "nombre": df["nombre"],
})
tabla_equivalencias.head(4)

,cliente_id,seudonimo,nombre
0,DM-00865,D4C5AC809820,Luis Castro
1,DM-00714,DBD3EE164CF6,Paula Beltran
2,DM-00572,33E091D5A353,Alvaro Riveros
3,DM-00737,ED774388A041,Julian Rojas


::: {.rds-card .decision}
**Pregunta 7.** Mientras exista la tabla anterior, los datos siguen siendo datos personales.

Y la tabla **tiene que existir**, porque sin ella la empresa no puede contactar al cliente que el modelo señaló. Ese es el nudo: la finalidad del proyecto —actuar sobre personas concretas— es incompatible con la anonimización real.

¿Dónde debería vivir esa tabla y quién debería poder leerla?
:::

::: {.callout-note collapse="true"}
## Respuesta orientativa (versión docente)

En un sistema separado del entorno de análisis, con control de acceso por rol y registro de consultas. El equipo de analítica trabaja con seudónimos; solo el proceso que ejecuta la campaña resuelve la equivalencia, y queda traza de cada resolución. La diferencia entre las tres nociones del curso: **minimizar** es no traer lo que no se necesita; **seudonimizar** es sustituir el identificador conservando la reversibilidad; **anonimizar** es lograr que nadie sea reidentificable con medios razonables, y se mide.
:::

---

# 5 · JUSTIFICAR

*¿Con qué evidencia?*

Una decisión sin evidencia no cuenta como decisión. Cerramos escribiendo el
registro del laboratorio.

In [14]:
registro = pd.DataFrame([{
    "proyecto": "DataMarket · modelo de abandono",
    "fecha": str(pd.Timestamp("today").date()),
    "variables_originales": df.shape[1],
    "variables_conservadas": df_min.shape[1],
    "criterio_minimizacion": ("solo comportamiento de compra y navegación; "
                              "sin demografía ni proxies de datos sensibles"),
    "umbral_riesgo_aceptado": "k >= 5 para al menos el 95 % de los registros",
    "registros_sin_evidencia_origen": int(df["fecha_autorizacion"].isna().sum()),
    "decision_sobre_esos_registros": "aislados, no borrados; excluidos del entrenamiento",
    "responsable": "equipo de analítica + Dirección Comercial",
}])

registro.to_csv("registro_lab01.csv", index=False, encoding="utf-8")
registro.T.rename(columns={0: "valor"})

,valor
proyecto,DataMarket · modelo de abandono
fecha,2026-09-22
variables_originales,35
variables_conservadas,8
criterio_minimizacion,solo comportamiento de compra y navegación; si...
umbral_riesgo_aceptado,k >= 5 para al menos el 95 % de los registros
registros_sin_evidencia_origen,236
decision_sobre_esos_registros,"aislados, no borrados; excluidos del entrenami..."
responsable,equipo de analítica + Dirección Comercial


::: {.rds-card .decision}
**Pregunta 8.** Suponga que esta base se va a compartir con un proveedor externo de analítica. Con lo que midió hoy, ¿la entregaría?

Si su respuesta es «sí, con condiciones», enumere las condiciones.
:::

::: {.callout-note collapse="true"}
## Respuesta orientativa (versión docente)

Tres salidas son defendibles: (1) entregar la versión generalizada, con el riesgo medido y aceptado por escrito; (2) entregar la seudonimizada bajo contrato de encargo del tratamiento, ambiente controlado, prohibición de cruce y plazo de destrucción; (3) no entregarla y ofrecer que el proveedor entrene dentro del perímetro, o entregar datos sintéticos. Lo que no es defendible es entregarla sin medir.
:::

---

# 6 · REFLEXIONAR

*¿Y en mi trabajo?*

::: {.rds-card .reflexiona}
**Pregunta 9 · de salida.** Tome una base con la que trabaje realmente. Ejecute mentalmente el bloque 3 sobre ella: ¿cuántas variables cuasi-identificadoras tiene?

> **¿Qué tendría que cambiar en mi proyecto de datos?**

Tres líneas. Nombre una variable concreta y un cambio ejecutable.
:::

::: {.callout-note collapse="true"}
## Respuesta orientativa (versión docente)

Lo que se busca no es una reflexión general sino un cambio nombrado: «voy a dejar de cargar `documento` en el dataset de modelado», «voy a medir k antes de entregar el tablero al área comercial». Si la respuesta es «ser más cuidadoso», no cuenta.
:::

---

## Lo que hicimos

1. Exploramos el tamaño real del problema: 1 512 registros × 35 variables.
2. Descubrimos que los faltantes de `fecha_autorizacion` no eran ruido sino la
   huella del origen de los datos.
3. Clasificamos las variables por riesgo, no por tipo.
4. Medimos que cuatro cuasi-identificadores dejan sola a la gran mayoría de las
   personas de la base.
5. Construimos una base minimizada para una finalidad declarada.
6. Seudonimizamos y vimos por qué eso no anonimiza.
7. Generalizamos y **medimos** cuánto bajó el riesgo, y cuánta utilidad costó.

**Siguiente paso:** Laboratorio 2 · auditoría de un pipeline completo, y el
dilema entre un modelo que predice mejor y uno que usa menos.

**Marco legal relacionado:** la excepción para fines estadísticos y científicos
del art. 10 de la Ley 1581 de 2012 exige suprimir la identidad de los titulares.
Este laboratorio muestra por qué eso no es trivial.